# Caso 1 · Pedido a fábrica — Limpieza de datos

Lee el Excel original, corrige sus problemas de calidad y guarda la base limpia (`data/processed/caso1_limpio.pkl`) que usan `01_eda` y `02_pedido_fabrica`.

## 1. Contexto del caso
- **Pregunta:** ¿cuántas unidades de cada modelo pedir a fábrica en la ventana que cierra el 30-oct-2026?
- **Decisión:** pedido trimestral de ~USD 11 millones que define lo que se venderá de febrero a mayo de 2027.
- **Éxito:** menos quiebres y menos venta perdida por disponibilidad (hoy 20 %), sin más unidades de más de 180 días en patio.
- **Restricciones:** mínimos y múltiplos por modelo, lead time de ~3.5 meses, última ventana de RAV4, corte de datos al 30-sep-2026.
- **Hipótesis:** la venta subestima la demanda de los modelos que quebraron y el inventario está desbalanceado por versión más que por volumen.

## 2. Datos disponibles
Un Excel con siete hojas (tabla siguiente). **Faltan y se pedirían:** inventario por VIN, ETA original por línea de pedido, pedidos de flota de 2027, plan comercial y tipo de cambio de las facturas en córdobas.


In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Rutas (el notebook corre desde notebooks/caso1/) y parámetros del ciclo; los leen 01_eda y 02_pedido_fabrica
ROOT = Path.cwd().resolve().parents[1]
assert (ROOT / "pyproject.toml").is_file(), (
    f"Ejecuta el notebook desde notebooks/caso1/ (raíz detectada: {ROOT})"
)
DATA_CASE1 = ROOT / "data" / "raw" / "caso1_vehiculos_nuevos.xlsx"
PROCESSED_CASE1 = ROOT / "data" / "processed" / "caso1_limpio.pkl"
PARAMS = {
    "cutoff_date": pd.Timestamp("2026-09-30"),  # fecha de corte de los datos
    "order_window_close": pd.Timestamp("2026-10-30"),  # cierre de la ventana de pedido
    "nio_per_usd": 36.6,  # tipo de cambio aproximado 2024-2026, solo para diagnosticar precios en córdobas
}
NIO_PER_USD = PARAMS["nio_per_usd"]
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Parámetros de limpieza (ver §3 Supuestos)
FLEET_THRESHOLD = 10  # S1.3 · ver §3
PRICE_RATIO_OK = (0.5, 2.0)  # S1.4 · ver §3
NO_AVAILABILITY_PATTERN = re.compile(r"disponib|stock|inventario|unidades", re.IGNORECASE)  # S1.6 · ver §3
ORDER_WINDOW_MONTHS = [1, 4, 7, 10]  # ventanas trimestrales de pedido (comunicado 2024-02-12)
BRANCHES = ["Managua Central", "Managua Sur", "León", "Chinandega", "Estelí", "Juigalpa", "Matagalpa"]
BRANCH_ALIASES = {"mga sur": "Managua Sur", "mga central": "Managua Central"}

raw = pd.read_excel(DATA_CASE1, sheet_name=None)
economics = raw["Economía por modelo"].copy()
MODELOS = economics["Modelo"].tolist()  # catálogo oficial: los 11 modelos de la hoja Economía
pd.DataFrame({k: v.shape for k, v in raw.items()}, index=["filas", "columnas"]).T

,filas,columnas
Diccionario,37,3
Ventas,4683,12
Inventario,4062,6
Leads,12132,13
Pedidos,2258,9
Economía por modelo,11,6
Comunicados,5,4


## 3. Supuestos
Decisiones que condicionan los resultados de este notebook y por qué se tomaron. En el texto y en el código se citan por su código.

| Código | Supuesto | Por qué | Dónde se aplica |
|---|---|---|---|
| **S1.1** | Las filas duplicadas exactas de *Ventas* se eliminan. | Son copias de la misma factura (mismo número y mismos datos); contarlas dos veces inflaría la demanda. | §4.2 |
| **S1.2** | Las notas de crédito (`NC-…`) restan una unidad de la demanda, pero la unidad **no vuelve al inventario**. | Todas referencian una factura real del mismo modelo y versión, 4 a 24 días después: son devoluciones. La conciliación de inventario muestra que la hoja *Inventario* no devuelve la unidad al patio. | §4.2 |
| **S1.3** | Una línea de factura u oportunidad con 10 o más unidades (`FLEET_THRESHOLD = 10`) es flota especial y sale de la demanda base. | Las dos facturas (60 y 45 unidades) entraron por pedidos especiales fuera de ventana, como prevé el comunicado de fábrica; no salen del inventario normal. No hay líneas entre 5 y 44 unidades, así que el umbral exacto no cambia el resultado. | §4.3 |
| **S1.4** | Un precio fuera de 0.5–2.0 veces la mediana del modelo (`PRICE_RATIO_OK`) se excluye del análisis de precio; se convierte a 36.6 C$/USD (`NIO_PER_USD`) solo como referencia. | Esas 13 líneas están ~36 veces sobre la mediana de su versión, que es el tipo de cambio: se capturaron en córdobas. Como no se conoce el tipo de cambio de cada factura, se excluyen. Las unidades no cambian. | §4.4 |
| **S1.5** | Las combinaciones de inventario sin antigüedad se imputan con el promedio del modelo en ese mes (`age_days`). | Son 40 combinaciones y 104 unidades (<1 %); el promedio del modelo-mes es la mejor referencia disponible y no mueve los tramos de antigüedad. | §4.5 |
| **S1.6** | Las redacciones "sin disponibilidad / sin stock / sin inventario / versión o color no disponible" son un solo motivo de pérdida (`NO_AVAILABILITY_PATTERN`). | Describen la misma causa escrita en texto libre de ocho formas; separarlas subestimaría la demanda perdida por falta de stock. | §4.6 |
| **S1.11** | Los IDs de oportunidad repetidos en *Leads* se eliminan y se conserva la primera fila. | Las 60 filas repetidas son idénticas en todos los campos: son cargas duplicadas, no oportunidades distintas. | §4.6 |

## 4. Validación y limpieza

### 4.1 Nombres sin estandarizar
Modelo y sucursal se normalizan contra el catálogo de *Economía por modelo* y la lista de sucursales.

In [2]:
vc = raw["Ventas"]["Modelo"].value_counts()
print(
    f"Modelo en Ventas tal como viene: {vc.size} valores distintos para {len(MODELOS)} modelos reales. Ejemplos:"
)
print(vc.iloc[10:26].to_string())
print("\nSucursal en Ventas:", sorted(raw["Ventas"]["Sucursal"].unique()))

Modelo en Ventas tal como viene: 52 valores distintos para 11 modelos reales. Ejemplos:
Modelo
Land Cruiser Prado      116
Yaris                     5
Corolla  Cross            5
Hilux                     5
Toyota Hilux              5
Yaris  Cross              4
rav4                      4
HILUX                     3
Corolla Cross             3
 Raize                    3
Toyota Corolla Cross      3
Fortuner                  2
YARIS                     2
RAIZE                     2
 Hilux                    2
Toyota Yaris Cross        2

Sucursal en Ventas: ['Chinandega', 'Esteli', 'Estelí', 'Juigalpa', 'Leon', 'León', 'MANAGUA CENTRAL', 'MATAGALPA', 'Managua Central', 'Managua Sur', 'Matagalpa', 'Mga Sur']


In [3]:
def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))


def collapse_spaces(x):
    return x if pd.isna(x) else " ".join(str(x).split())


CATALOG = {m.lower(): m for m in MODELOS}
BRANCH_MAP = {strip_accents(b).lower(): b for b in BRANCHES} | BRANCH_ALIASES


def normalize_model(x):
    # Quita espacios extra y el prefijo "Toyota", y unifica mayúsculas contra el catálogo
    if pd.isna(x):
        return x
    s = re.sub(r"(?i)^toyota\s+", "", collapse_spaces(x))
    return CATALOG.get(s.lower(), s)


def normalize_branch(x):
    if pd.isna(x):
        return x
    return BRANCH_MAP.get(strip_accents(collapse_spaces(x)).lower(), x)


sheets = {k: v.copy() for k, v in raw.items()}
log = []


def apply(sheet, col, fn):
    before = sheets[sheet][col]
    after = before.map(fn)
    changed = int((before.fillna("∅") != after.fillna("∅")).sum())
    log.append((sheet, col, before.nunique(), after.nunique(), changed))
    sheets[sheet][col] = after


for sheet in ["Ventas", "Inventario", "Pedidos"]:
    apply(sheet, "Modelo", normalize_model)
    for col in ["Versión", "Color"]:
        apply(sheet, col, collapse_spaces)
for col in ["Modelo de interés", "Modelo comprado (si difiere)"]:
    apply("Leads", col, normalize_model)
for sheet in ["Ventas", "Leads"]:
    apply(sheet, "Sucursal", normalize_branch)

changes = pd.DataFrame(
    log, columns=["Hoja", "Columna", "Variantes antes", "Variantes después", "Celdas cambiadas"]
)
display(changes[changes["Celdas cambiadas"] > 0].reset_index(drop=True))

# Comprobación: tras normalizar, nada queda fuera de catálogo
for sheet, col in [
    ("Ventas", "Modelo"),
    ("Inventario", "Modelo"),
    ("Pedidos", "Modelo"),
    ("Leads", "Modelo de interés"),
]:
    fuera = set(sheets[sheet][col].dropna()) - set(MODELOS)
    assert not fuera, f"{sheet}.{col} fuera de catálogo: {fuera}"
for sheet in ["Ventas", "Leads"]:
    assert set(sheets[sheet]["Sucursal"]) <= set(BRANCHES), sheet
print("Todas las hojas quedan en los 11 modelos y las 7 sucursales oficiales.")

,Hoja,Columna,Variantes antes,Variantes después,Celdas cambiadas
0,Ventas,Modelo,52,11,87
1,Inventario,Modelo,42,11,46
2,Pedidos,Modelo,37,11,47
3,Leads,Modelo de interés,63,11,164
4,Ventas,Sucursal,12,7,13


Todas las hojas quedan en los 11 modelos y las 7 sucursales oficiales.


Modelo y sucursal tenían variantes de captura (mayúsculas, espacios, prefijo *Toyota*); versión y color venían limpios. Tras normalizar quedan los 11 modelos y 7 sucursales oficiales.

### 4.2 Duplicados y notas de crédito

In [4]:
raw_v = sheets["Ventas"]
print(
    f"Filas duplicadas exactas en Ventas: {raw_v.duplicated().sum()} "
    f"(= facturas repetidas: {raw_v['Factura'].duplicated().sum()}; no hay facturas repetidas con datos distintos)"
)
display(raw_v[raw_v.duplicated(keep=False)].sort_values("Factura").head(4))

sales = raw_v.drop_duplicates().copy()  # S1.1 · ver §3
sales["is_credit_note"] = sales["Factura"].str.startswith("NC")  # S1.2 · ver §3

nc = sales[sales["is_credit_note"]].copy()
nc["factura_ref"] = nc["Nota"].str.extract(r"(FAC-\d+)")[0]
orig = sales.set_index("Factura")
nc["dias_desde_venta"] = (nc["Fecha"] - orig.loc[nc["factura_ref"], "Fecha"].values).dt.days
nc["coincide_modelo_version"] = (nc["Modelo"].values == orig.loc[nc["factura_ref"], "Modelo"].values) & (
    nc["Versión"].values == orig.loc[nc["factura_ref"], "Versión"].values
)
print(
    f"\nNotas de crédito: {len(nc)} líneas, {nc['Unidades'].sum()} unidades. Todas referencian una factura existente, "
    f"del mismo modelo y versión: {nc['coincide_modelo_version'].all()}"
)
nc[["Fecha", "Factura", "Modelo", "Versión", "Unidades", "factura_ref", "dias_desde_venta"]]

Filas duplicadas exactas en Ventas: 18 (= facturas repetidas: 18; no hay facturas repetidas con datos distintos)


,Fecha,Factura,Sucursal,Vendedor,Tipo de cliente,Cliente,Modelo,Versión,Color,Unidades,Precio unitario (USD),Nota
65,2023-10-16,FAC-880066,Managua Central,Fátima Urbina,Individual,Claudia Talavera Flores,Fortuner,4x4 AT,Plata Metálico,1,56500,NaN
66,2023-10-16,FAC-880066,Managua Central,Fátima Urbina,Individual,Claudia Talavera Flores,Fortuner,4x4 AT,Plata Metálico,1,56500,NaN
200,2023-11-14,FAC-880200,Managua Central,Fátima Urbina,Individual,Félix Rodríguez Ramírez,Corolla Cross,SEG 2.0 CVT,Gris Grafito,1,32590,NaN
201,2023-11-14,FAC-880200,Managua Central,Fátima Urbina,Individual,Félix Rodríguez Ramírez,Corolla Cross,SEG 2.0 CVT,Gris Grafito,1,32590,NaN



Notas de crédito: 9 líneas, -9 unidades. Todas referencian una factura existente, del mismo modelo y versión: True


,Fecha,Factura,Modelo,Versión,Unidades,factura_ref,dias_desde_venta
477,2023-12-30,NC-880383,Hilux,Doble Cabina 4x4 MT,-1,FAC-880383,15
583,2024-01-30,NC-880562,Hilux,Cabina Sencilla 4x4 MT,-1,FAC-880562,5
1159,2024-06-08,NC-881095,Hilux,SRV 4x4 AT,-1,FAC-881095,16
3682,2026-01-03,NC-883646,RAV4,XLE 2WD,-1,FAC-883646,4
3744,2026-01-23,NC-883650,Hilux,Doble Cabina 4x4 MT,-1,FAC-883650,24
3825,2026-02-13,NC-883736,Corolla Cross,SEG 2.0 CVT,-1,FAC-883736,19
3997,2026-03-21,NC-883893,Hilux,Cabina Sencilla 4x4 MT,-1,FAC-883893,15
4275,2026-05-29,NC-884204,Yaris Cross,SEG Híbrido,-1,FAC-884204,12
4471,2026-08-03,NC-884397,RAV4,XLE AWD,-1,FAC-884397,15


- 18 duplicados exactos se eliminan.
- 9 notas de crédito son devoluciones reales: restan una unidad de la demanda, pero la unidad no vuelve al inventario (se comprueba en §4.8).

*Supuestos S1.1, S1.2 (ver §3).*

### 4.3 Facturas de flota especial

In [5]:
sales["special_fleet"] = sales["Unidades"] >= FLEET_THRESHOLD  # S1.3 · ver §3
print("Distribución de unidades por línea de factura:")
print(sales["Unidades"].value_counts().sort_index().to_frame("líneas").T.to_string())
print(
    "\nLíneas de 'Flota / Institucional':",
    (sales["Tipo de cliente"] == "Flota / Institucional").sum(),
    "| de ellas con ≥",
    FLEET_THRESHOLD,
    "unidades:",
    sales["special_fleet"].sum(),
)
cols = ["Fecha", "Factura", "Sucursal", "Tipo de cliente", "Cliente", "Modelo", "Versión", "Unidades"]
display(sales.loc[sales["special_fleet"], cols])

ped = sheets["Pedidos"]
fuera_ventana = ped[~ped["Fecha pedido"].dt.month.isin(ORDER_WINDOW_MONTHS)]
print("\nPedidos a fábrica fuera de las ventanas trimestrales:")
fuera_ventana[["No. pedido", "Fecha pedido", "Modelo", "Versión", "Unidades", "Fecha de recepción", "Estado"]]

Distribución de unidades por línea de factura:
Unidades  -1     1    2    3    4    45   60
líneas      9  4332  223   85   14    1    1

Líneas de 'Flota / Institucional': 152 | de ellas con ≥ 10 unidades: 2


,Fecha,Factura,Sucursal,Tipo de cliente,Cliente,Modelo,Versión,Unidades
3976,2026-03-18,FAC-883958,Managua Central,Flota / Institucional,Ministerio de Infraestructura Rural (ficticio),Hilux,Doble Cabina 4x4 MT,60
4310,2026-06-10,FAC-884287,Managua Central,Flota / Institucional,Autorentas del Pacífico S.A. (ficticio),Yaris,Sedán XLI MT,45



Pedidos a fábrica fuera de las ventanas trimestrales:


,No. pedido,Fecha pedido,Modelo,Versión,Unidades,Fecha de recepción,Estado
1735,PF-41012,2025-11-25,Hilux,Doble Cabina 4x4 MT,60,2026-03-03,Recibido
1908,PF-41013,2026-02-25,Yaris,Sedán XLI MT,45,2026-06-03,Recibido


Dos facturas de 60 y 45 unidades entraron por pedidos especiales fuera de ventana (PF-41012 y PF-41013). No son demanda de inventario y se excluyen; las demás ventas a flotas (1 a 4 unidades) se mantienen.

*Supuesto S1.3 (ver §3).*

### 4.4 Precios capturados en otra moneda

In [6]:
model_median = sales.groupby("Modelo")["Precio unitario (USD)"].transform("median")
sales["price_ratio"] = sales["Precio unitario (USD)"] / model_median
sales["price_in_cordobas"] = ~sales["price_ratio"].between(*PRICE_RATIO_OK)  # S1.4 · ver §3
sales["price_usd"] = sales["Precio unitario (USD)"].where(~sales["price_in_cordobas"])
sales["price_usd_converted"] = sales["price_usd"].fillna(sales["Precio unitario (USD)"] / NIO_PER_USD)

out = sales[sales["price_in_cordobas"]].copy()
version_median = sales[~sales["price_in_cordobas"]].groupby(["Modelo", "Versión"])["price_usd"].median()
out["FX implícito vs mediana de la versión"] = (
    out["Precio unitario (USD)"]
    / version_median.reindex(pd.MultiIndex.from_frame(out[["Modelo", "Versión"]])).values
)
print(
    f"Líneas con precio fuera de [0.5x, 2x] la mediana del modelo: {len(out)}; "
    f"ratio entre {out['price_ratio'].min():.0f}x y {out['price_ratio'].max():.0f}x la mediana del modelo, "
    f"y entre {out['FX implícito vs mediana de la versión'].min():.1f} y {out['FX implícito vs mediana de la versión'].max():.1f} "
    f"frente a la mediana de su versión (tipo de cambio NIO/USD de 2024-2026 ≈ {NIO_PER_USD})."
)
ok = sales.loc[~sales["price_in_cordobas"], "price_ratio"]
print(
    f"El resto de los precios está entre {ok.min():.2f}x y {ok.max():.2f}x la mediana del modelo: no hay más atípicos."
)
out[
    [
        "Fecha",
        "Factura",
        "Modelo",
        "Versión",
        "Unidades",
        "Precio unitario (USD)",
        "price_ratio",
        "FX implícito vs mediana de la versión",
    ]
].sort_values("price_ratio")

Líneas con precio fuera de [0.5x, 2x] la mediana del modelo: 13; ratio entre 32x y 44x la mediana del modelo, y entre 35.4 y 37.2 frente a la mediana de su versión (tipo de cambio NIO/USD de 2024-2026 ≈ 36.6).
El resto de los precios está entre 0.73x y 1.21x la mediana del modelo: no hay más atípicos.


,Fecha,Factura,Modelo,Versión,Unidades,Precio unitario (USD),price_ratio,FX implícito vs mediana de la versión
2904,2025-07-18,FAC-882892,Hilux,Doble Cabina 4x2 MT,1,1365560,31.92,36.16
1870,2024-11-25,FAC-881859,Fortuner,4x2 AT,1,1789253,32.29,36.73
1240,2024-06-27,FAC-881231,Hilux,Doble Cabina 4x2 MT,1,1387898,32.44,36.76
3000,2025-08-12,FAC-882988,Corolla Cross,SEG 2.0 CVT,1,1189051,36.36,36.22
1109,2024-05-25,FAC-881101,Rush,G 1.5 AT,1,939303,36.81,36.20
2740,2025-06-10,FAC-882729,Hilux,Doble Cabina 4x4 MT,1,1581618,36.97,36.66
1755,2024-10-30,FAC-881744,Fortuner,4x4 AT,1,2069030,37.33,37.10
3244,2025-10-12,FAC-883232,Corolla Cross,SEG 2.0 CVT,1,1222376,37.38,37.24
4255,2026-05-24,FAC-884235,Rush,G 1.5 AT,1,966402,37.87,37.24
1091,2024-05-20,FAC-881083,Yaris,Hatchback S AT,1,810767,38.84,37.06


13 líneas tienen precios ~36 veces la mediana de su versión: es el tipo de cambio, se capturaron en córdobas. Se excluyen del análisis de precio; las unidades no cambian.

*Supuesto S1.4 (ver §3).*

### 4.5 Nulos y antigüedad del inventario

In [7]:
for hoja in ["Ventas", "Inventario", "Leads", "Pedidos"]:
    nulos = sheets[hoja].isna().sum()
    print(f"{hoja:<11}", nulos[nulos > 0].to_dict() or "sin nulos")

inventory = sheets["Inventario"].copy()
inventory["month"] = inventory["Mes (cierre)"].dt.to_period("M")
col_edad = "Antigüedad promedio (días)"
# S1.5 · ver §3
inventory["age_days"] = inventory[col_edad].fillna(
    inventory.groupby(["month", "Modelo"])[col_edad].transform("mean")
)
print(
    f"\nAntigüedad nula: {inventory[col_edad].isna().sum()} combinaciones "
    f"({inventory.loc[inventory[col_edad].isna(), 'Unidades'].sum():.0f} unidades); tras imputar: {inventory['age_days'].isna().sum()} nulos"
)

Ventas      {'Color': 18, 'Nota': 4674}
Inventario  {'Antigüedad promedio (días)': 40}
Leads       {'Versión de interés': 145, 'Motivo de pérdida': 4532, 'Modelo comprado (si difiere)': 12040, 'Versión comprada (si difiere)': 11735}
Pedidos     {'Fecha de recepción': 182}

Antigüedad nula: 40 combinaciones (104 unidades); tras imputar: 0 nulos


Los nulos son los esperables (la nota solo existe en notas de crédito, la fecha de recepción solo falta en tránsito). `Color` nulo (18 líneas) queda como *desconocido*. Solo la antigüedad del inventario requiere imputación.

*Supuesto S1.5 (ver §3).*

### 4.6 Leads (CRM)

In [8]:
raw_l = sheets["Leads"]
dups = raw_l[raw_l["ID Oportunidad"].duplicated(keep=False)]
print(
    f"Leads: {len(raw_l)} filas; IDs de oportunidad repetidos: {raw_l['ID Oportunidad'].duplicated().sum()} "
    f"(con contenido distinto: {dups.drop_duplicates().shape[0] - dups['ID Oportunidad'].nunique()})"
)
leads = raw_l.drop_duplicates(subset="ID Oportunidad").copy()  # S1.11 · ver §3
leads["special_fleet"] = leads["Unidades"] >= FLEET_THRESHOLD  # S1.3 · ver §3, mismo criterio que en Ventas
leads["no_availability"] = leads["Motivo de pérdida"].str.contains(
    NO_AVAILABILITY_PATTERN, na=False
)  # S1.6 · ver §3
leads["loss_reason_group"] = np.where(
    leads["no_availability"], "Sin disponibilidad (agrupado)", leads["Motivo de pérdida"]
)
leads["month"] = leads["Fecha cierre"].dt.to_period("M")

print(
    f"Resultado: {leads['Resultado'].value_counts().to_dict()} | Fecha cierre < creación: {(leads['Fecha cierre'] < leads['Fecha creación']).sum()}"
)
print(f"Oportunidades de flota especial (≥ {FLEET_THRESHOLD} u.): {leads['special_fleet'].sum()}")
print("\nMotivos de pérdida tal como vienen (✔ = agrupado como 'sin disponibilidad'):")
motivos = leads["Motivo de pérdida"].value_counts().to_frame("leads")
motivos["agrupado"] = np.where(motivos.index.str.contains(NO_AVAILABILITY_PATTERN), "✔", "")
motivos

Leads: 12132 filas; IDs de oportunidad repetidos: 60 (con contenido distinto: 0)
Resultado: {'Perdida': 7563, 'Ganada': 4509} | Fecha cierre < creación: 0
Oportunidades de flota especial (≥ 10 u.): 2

Motivos de pérdida tal como vienen (✔ = agrupado como 'sin disponibilidad'):


,leads,agrupado
Motivo de pérdida,,
No responde / sin seguimiento,1811,
Precio,1653,
Financiamiento rechazado,1449,
Pospone la compra,1086,
Compró a la competencia,1032,
Sin disponibilidad / sin inventario,95,✔
Sin disponibilidad,82,✔
Sin inventario,75,✔
No hay unidades disponibles,72,✔


Los motivos de pérdida por falta de stock venían en ocho redacciones y se agrupan en uno. Tres de ellas (137 leads) hablan de versión o color, no del modelo: el EDA lo retoma. Las dos oportunidades de flota se marcan igual que sus facturas.

*Supuestos S1.6, S1.11 (ver §3).*

### 4.7 Pedidos y reglas de fábrica

In [9]:
orders = sheets["Pedidos"].copy()
orders["order_month"] = orders["Fecha pedido"].dt.to_period("M")
orders["order_quarter"] = orders["Fecha pedido"].dt.to_period("Q")
orders["in_order_window"] = orders["Fecha pedido"].dt.month.isin(ORDER_WINDOW_MONTHS)
communications = sheets["Comunicados"]


def parse_order_rules(communications: pd.DataFrame) -> pd.DataFrame:
    # Pedido mínimo y múltiplo por modelo, leídos del comunicado de fábrica del 2025-06-03
    text = communications.loc[communications["Asunto"].str.contains("mínimo", case=False), "Texto"].iloc[0]
    body = text.split("múltiplo):")[1]
    pairs = []
    for chunk in re.split(r"[;.]", body):
        m = re.match(r"\s*([A-Za-z][A-Za-z0-9 ]+?)\s+(\d+)\s*/\s*(\d+)\s*$", chunk)
        if m:
            pairs.append(m.groups())
    rules = pd.DataFrame(pairs, columns=["Modelo", "min_order", "lot_multiple"])
    rules[["min_order", "lot_multiple"]] = rules[["min_order", "lot_multiple"]].astype(int)
    return rules.set_index("Modelo")


rules = parse_order_rules(communications)
assert set(rules.index) == set(MODELOS), set(MODELOS) ^ set(rules.index)
print(
    f"Pedidos: {len(orders)} líneas, {orders['No. pedido'].nunique()} pedidos; estados: {orders['Estado'].value_counts().to_dict()}"
)
print(
    "Fecha de recepción nula solo en tránsito:",
    orders.loc[orders["Fecha de recepción"].isna(), "Estado"].eq("En tránsito").all(),
)
rules.rename(columns={"min_order": "Pedido mínimo", "lot_multiple": "Múltiplo"}).T

Pedidos: 2258 líneas, 145 pedidos; estados: {'Recibido': 2076, 'En tránsito': 182}
Fecha de recepción nula solo en tránsito: True


Modelo,Hilux,Fortuner,Corolla,Corolla Cross,Yaris,Yaris Cross,RAV4,Land Cruiser Prado,Rush,Hiace,Raize
Pedido mínimo,12,8,6,12,16,12,4,2,8,3,12
Múltiplo,6,4,6,6,8,6,4,2,4,3,6


Las 11 reglas de mínimo y múltiplo leídas del comunicado coinciden con el catálogo. La fecha de recepción solo falta en las líneas en tránsito.

### 4.8 Coherencia entre hojas: ¿cuadra el inventario?
Prueba: stock del mes = stock anterior + recepciones − facturas.

In [10]:
def monthly_table(df, model_col, value_col, months=None):
    # Pivote mes × modelo con ceros donde no hay registro (las hojas solo traen combinaciones con dato)
    t = df.pivot_table(index="month", columns=model_col, values=value_col, aggfunc="sum", fill_value=0)
    t = t.reindex(columns=MODELOS, fill_value=0)
    if months is None:
        months = pd.period_range(t.index.min(), t.index.max(), freq="M")
    return t.reindex(months, fill_value=0)


sales["month"] = sales["Fecha"].dt.to_period("M")
sales["fiscal_year"] = np.where(
    sales["Fecha"].dt.month >= 10, sales["Fecha"].dt.year + 1, sales["Fecha"].dt.year
)

stock = monthly_table(inventory, "Modelo", "Unidades")
recibidos = orders[orders["Estado"] == "Recibido"].assign(
    month=lambda x: x["Fecha de recepción"].dt.to_period("M")
)
received = monthly_table(recibidos, "Modelo", "Unidades", stock.index)
sold = monthly_table(
    sales[~sales["is_credit_note"]], "Modelo", "Unidades", stock.index
)  # todas las facturas, sin NC
dif = (stock - (stock.shift(1) + received - sold)).iloc[1:]
print(
    f"Sin contar las notas de crédito: celdas modelo-mes con diferencia entre stock reportado e implícito = "
    f"{int((dif != 0).sum().sum())} de {dif.size}."
)

# Si las notas de crédito devolvieran la unidad al patio, el stock implícito sería 1 unidad mayor en esos meses:
dif_con_nc = dif + monthly_table(sales[sales["is_credit_note"]], "Modelo", "Unidades", dif.index)
print(
    "Contando las notas de crédito como unidad devuelta al patio, aparece −1 exactamente en el mes de cada una:"
)
dif_con_nc[dif_con_nc.abs().sum(axis=1) > 0].astype(int)

Sin contar las notas de crédito: celdas modelo-mes con diferencia entre stock reportado e implícito = 0 de 385.
Contando las notas de crédito como unidad devuelta al patio, aparece −1 exactamente en el mes de cada una:


Modelo,Hilux,Fortuner,Corolla,Corolla Cross,Yaris,Yaris Cross,RAV4,Land Cruiser Prado,Rush,Hiace,Raize
2023-12,-1,0,0,0,0,0,0,0,0,0,0
2024-01,-1,0,0,0,0,0,0,0,0,0,0
2024-06,-1,0,0,0,0,0,0,0,0,0,0
2026-01,-1,0,0,0,0,0,-1,0,0,0,0
2026-02,0,0,0,-1,0,0,0,0,0,0,0
2026-03,-1,0,0,0,0,0,0,0,0,0,0
2026-05,0,0,0,0,0,-1,0,0,0,0,0
2026-08,0,0,0,0,0,0,-1,0,0,0,0


El inventario **cuadra exactamente** en los 385 modelo-mes: no hay ventas ni pedidos sin registrar, y las 105 unidades de flota especial sí pasaron por el patio. Si la nota de crédito devolviera la unidad aparecería un −1 en cada uno de esos meses; no aparece, lo que confirma **S1.2**. Stock y tránsito son confiables para el modelo de pedido.

## 5. Base limpia y exportación
Se guarda en `data/processed/caso1_limpio.pkl`: tablas normalizadas, demanda base (venta neta sin flotas especiales), reglas de fábrica y parámetros del ciclo (corte 30-sep-2026, cierre de ventana 30-oct-2026). Las agregaciones mes × modelo se hacen en `02_pedido_fabrica` §3.

In [11]:
cambiadas = changes.set_index(["Hoja", "Columna"])["Celdas cambiadas"]
calidad = pd.DataFrame(
    [
        (
            "Ventas",
            "Modelo con variantes (mayúsculas, espacios, prefijo 'Toyota')",
            cambiadas[("Ventas", "Modelo")],
            "Normalizado contra catálogo",
        ),
        (
            "Ventas",
            "Sucursal con variantes (MANAGUA CENTRAL, Mga Sur, Leon…)",
            cambiadas[("Ventas", "Sucursal")],
            "Normalizado",
        ),
        ("Ventas", "Filas duplicadas exactas", raw["Ventas"].duplicated().sum(), "Eliminadas (S1.1)"),
        (
            "Ventas",
            "Notas de crédito (devoluciones, −1 unidad)",
            int(sales["is_credit_note"].sum()),
            "Se netean de la demanda; no vuelven a inventario (S1.2)",
        ),
        (
            "Ventas",
            f"Facturas de flota especial (≥ {FLEET_THRESHOLD} unidades)",
            int(sales["special_fleet"].sum()),
            "Excluidas de la demanda base (S1.3)",
        ),
        (
            "Ventas",
            "Precio capturado en córdobas (~32-44x)",
            int(sales["price_in_cordobas"].sum()),
            "Excluido del análisis de precio (S1.4)",
        ),
        ("Ventas", "Color nulo", raw["Ventas"]["Color"].isna().sum(), "Se mantiene como desconocido"),
        ("Inventario", "Modelo con variantes", cambiadas[("Inventario", "Modelo")], "Normalizado"),
        (
            "Inventario",
            "Antigüedad nula",
            raw["Inventario"][col_edad].isna().sum(),
            "Imputada con el promedio del modelo-mes en age_days (S1.5)",
        ),
        (
            "Leads",
            "Modelo de interés con variantes",
            cambiadas[("Leads", "Modelo de interés")],
            "Normalizado",
        ),
        (
            "Leads",
            "ID de oportunidad duplicado (filas idénticas)",
            raw["Leads"]["ID Oportunidad"].duplicated().sum(),
            "Eliminados (S1.11)",
        ),
        (
            "Leads",
            "Motivo 'sin disponibilidad' con 8 redacciones",
            int(leads["no_availability"].sum()),
            "Agrupado en un motivo (S1.6)",
        ),
        (
            "Leads",
            f"Oportunidades de flota especial (≥ {FLEET_THRESHOLD} unidades)",
            int(leads["special_fleet"].sum()),
            "Marcadas como en Ventas (S1.3)",
        ),
        ("Pedidos", "Modelo con variantes", cambiadas[("Pedidos", "Modelo")], "Normalizado"),
        (
            "Todas",
            "Inventario vs facturas y recepciones (sin NC)",
            "0 dif.",
            "Cuadra exacto: sin tratamiento",
        ),
    ],
    columns=["Hoja", "Problema", "Filas", "Tratamiento"],
)
display(calidad.style.hide(axis="index"))

base_limpia = {
    "models": MODELOS,
    "sales": sales,
    "base_sales": sales[
        ~sales["special_fleet"]
    ],  # demanda base: venta neta sin flotas especiales (S1.3 · ver §3)
    "inventory": inventory,
    "leads": leads,
    "orders": orders,
    "economics": economics.set_index("Modelo"),
    "communications": communications,
    "rules": rules,
    "changes": changes,
    "quality": calidad,
    "params": PARAMS,
}
destino = PROCESSED_CASE1
destino.parent.mkdir(parents=True, exist_ok=True)
pd.to_pickle(base_limpia, destino)
print(
    f"Base limpia escrita en {destino.relative_to(ROOT)}: "
    + ", ".join(f"{k} ({len(v)})" for k, v in base_limpia.items() if isinstance(v, pd.DataFrame))
)

Hoja,Problema,Filas,Tratamiento
Ventas,"Modelo con variantes (mayúsculas, espacios, prefijo 'Toyota')",87,Normalizado contra catálogo
Ventas,"Sucursal con variantes (MANAGUA CENTRAL, Mga Sur, Leon…)",13,Normalizado
Ventas,Filas duplicadas exactas,18,Eliminadas (S1.1)
Ventas,"Notas de crédito (devoluciones, −1 unidad)",9,Se netean de la demanda; no vuelven a inventario (S1.2)
Ventas,Facturas de flota especial (≥ 10 unidades),2,Excluidas de la demanda base (S1.3)
Ventas,Precio capturado en córdobas (~32-44x),13,Excluido del análisis de precio (S1.4)
Ventas,Color nulo,18,Se mantiene como desconocido
Inventario,Modelo con variantes,46,Normalizado
Inventario,Antigüedad nula,40,Imputada con el promedio del modelo-mes en age_days (S1.5)
Leads,Modelo de interés con variantes,164,Normalizado


Base limpia escrita en data/processed/caso1_limpio.pkl: sales (4665), base_sales (4663), inventory (4062), leads (12072), orders (2258), economics (11), communications (5), rules (11), changes (13), quality (15)


**Qué se exporta:** un diccionario de tablas que se lee con `pd.read_pickle`. Las columnas derivadas tienen nombre en inglés (`is_credit_note`, `special_fleet`, `price_usd`, `age_days`…); `base_sales` es la demanda base y `params` trae la fecha de corte y el cierre de la ventana.